# Episode 7: Gradient Descent: Walking Downhill
<a href="https://colab.research.google.com/github/Kinvecta/how-llms-work/blob/main/notebooks/Episode%2007%20-%20Gradient%20Descent.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

**Companion notebook** for the series *How Large Language Models Work: From Math to Transformers*.

This notebook reproduces the Math Lab numbers from Episode 7, one step per section. It uses only `numpy` (already installed in Colab), so choose **Runtime, Run all** and nothing else is needed.

Covers: the update θ ← θ − η∇L on the running sentence (N = 1), the factor 1 − 2ηκ, Episode 5's bowl, the 1/N scale of the mean loss, and mini-batch gradients (exact enumeration). All toy numbers are illustrative.

Every value that appears in the episode is checked with an `assert`. If a cell fails, a number in the script and the math have drifted apart.

In [1]:
import numpy as np
from itertools import product
print("numpy", np.__version__)

numpy 2.5.3


## Section 3: the slider on the sentence's loss
N = 1. L(z) = −ln p with p = e^z / (e^z + R), R = 0.70, starting at z = ln 0.30.

In [2]:
R = 0.70
def p_of(z):  return np.exp(z) / (np.exp(z) + R)
def loss(z):  return -np.log(p_of(z))
def grad(z):  return -(1 - p_of(z))          # dL/dz

z0 = np.log(0.30)
print(f"start: z = {z0:.3f}, p = {p_of(z0):.3f}, L = {loss(z0):.3f}, dL/dz = {grad(z0):.2f}")
assert round(p_of(z0), 3) == 0.300 and round(loss(z0), 3) == 1.204 and round(grad(z0), 2) == -0.70

# finite-difference check of the gradient
h = 1e-6
assert abs((loss(z0 + h) - loss(z0 - h)) / (2 * h) - grad(z0)) < 1e-6

def run(eta, steps, z=z0):
    for _ in range(steps):
        z = z - eta * grad(z)
    return z

# closed form after one step: p = 1 / (1 + (7/3) e^(-0.70 eta))
for eta in (0.1, 1, 5):
    z1 = run(eta, 1)
    assert abs(p_of(z1) - 1 / (1 + (7 / 3) * np.exp(-0.70 * eta))) < 1e-12

table = {0.1: ((0.315, 1.155), (0.447, 0.806)),
         1:   ((0.463, 0.770), (0.894, 0.112)),
         5:   ((0.934, 0.068), (0.984, 0.016))}
print("eta | after 1 step: p, L | after 10 steps: p, L")
for eta, (a1, a10) in table.items():
    z1, z10 = run(eta, 1), run(eta, 10)
    got1 = (round(p_of(z1), 3), round(loss(z1), 3))
    got10 = (round(p_of(z10), 3), round(loss(z10), 3))
    print(f"{eta:>3} | {got1} | {got10}")
    assert got1 == a1 and got10 == a10

# "nothing blows up": L falls at every step, for every eta tried (numerical check, not a proof)
for eta in np.concatenate([np.linspace(0.01, 10, 400), [100, 1000]]):
    z, last = z0, loss(z0)
    for _ in range(200):
        z = z - eta * grad(z)
        now = loss(z)
        assert now <= last + 1e-15
        last = now
print("L never rose, for 402 values of eta up to 1000 and 200 steps each.")

start: z = -1.204, p = 0.300, L = 1.204, dL/dz = -0.70
eta | after 1 step: p, L | after 10 steps: p, L
0.1 | (np.float64(0.315), np.float64(1.155)) | (np.float64(0.447), np.float64(0.806))
  1 | (np.float64(0.463), np.float64(0.77)) | (np.float64(0.894), np.float64(0.112))
  5 | (np.float64(0.934), np.float64(0.068)) | (np.float64(0.984), np.float64(0.016))
L never rose, for 402 values of eta up to 1000 and 200 steps each.


## Math Lab, Steps 1 and 2: one dimension
f(z) = κz², f′(z) = 2κz, so each step multiplies z by (1 − 2ηκ).

In [3]:
def steps_to(eta, kappa=1.0, z0=1.0, tol=0.01, cap=10**6):
    factor = 1 - 2 * eta * kappa
    z, k = z0, 0
    while abs(z) > tol and k < cap:
        z *= factor
        k += 1
    return k

for eta, want in ((0.01, 228), (0.25, 7), (0.5, 1)):
    print(f"eta = {eta}: {steps_to(eta)} steps to |z| <= 0.01")
    assert steps_to(eta) == want
assert abs(steps_to(0.01) * 0.01 - 2.3) < 0.05            # "about 2.3 / eta" for small eta

kappa = 1.0
# eta = 1/(2 kappa) lands on 0; eta = 1/kappa bounces between +/- z0; above it grows
z = 1.0
assert 1 - 2 * 0.5 * kappa == 0
z = 1.0
for _ in range(5):
    z *= 1 - 2 * 1.0 * kappa
assert z == -1.0
z = 1.0
for _ in range(5):
    z *= 1 - 2 * 1.2 * kappa
assert abs(z) > 1
print("stable range 0 < eta < 1/kappa, checked at 0.5, 1.0 and 1.2 (kappa = 1).")

eta = 0.01: 228 steps to |z| <= 0.01
eta = 0.25: 7 steps to |z| <= 0.01
eta = 0.5: 1 steps to |z| <= 0.01
stable range 0 < eta < 1/kappa, checked at 0.5, 1.0 and 1.2 (kappa = 1).


## Math Lab, Step 3: Episode 5's bowl
f = z₁² + 2z₂², ∇f = (2z₁, 4z₂). Start at (1, 1).

In [4]:
def bowl_f(z):  return z[0] ** 2 + 2 * z[1] ** 2
def bowl_g(z):  return np.array([2 * z[0], 4 * z[1]])

def bowl_run(eta, k):
    z = np.array([1.0, 1.0]); out = [bowl_f(z)]
    for _ in range(k):
        z = z - eta * bowl_g(z); out.append(bowl_f(z))
    return out

want = {0.1: [3, 1.36, 0.669, 0.355],
        1/3: [3, 0.333, 0.037, 0.004],
        0.5: [3, 2, 2, 2],
        1.0: [3, 19, 163, 1459]}
for eta, w in want.items():
    got = [round(v, 3) for v in bowl_run(eta, 3)]
    print(f"eta = {eta:.3f}: {got}")
    assert got == [round(v, 3) for v in w], (got, w)

# exact: eta = 1/3 gives 3 / 9^k
for k, v in enumerate(bowl_run(1/3, 4)):
    assert abs(v - 3 / 9 ** k) < 1e-12

got = [round(v, 3) for v in bowl_run(0.52, 2)]
print("eta = 0.52:", got)
assert got == [3, 2.334, 2.721]

# best repeated step: equalize the factors in size, 1 - 2 eta = -(1 - 4 eta) -> eta = 1/3, factor 1/3
assert abs((1 - 2/3) - (1/3)) < 1e-12
z = np.array([1.0, 1.0]); k = 0
while max(abs(z)) >= 0.01:
    z = z - (1/3) * bowl_g(z); k += 1
print("steps for both coordinates below 0.01 at eta = 1/3:", k)
assert k == 5

# Episode 5's single-step numbers: limit 0.556 (= 40/72) and best single step 0.278 (= 20/72)
f1 = lambda eta: bowl_f(np.array([1.0, 1.0]) - eta * bowl_g(np.array([1.0, 1.0])))
assert abs(f1(40/72) - 3) < 1e-9 and f1(0.55) < 3 < f1(0.57)
best = min(np.linspace(0, 0.5, 50001), key=f1)
print(f"best single step eta = {best:.3f}, f = {f1(best):.3f}")
assert round(best, 3) == 0.278 and round(f1(best), 3) == 0.222

eta = 0.100: [np.float64(3.0), np.float64(1.36), np.float64(0.669), np.float64(0.355)]
eta = 0.333: [np.float64(3.0), np.float64(0.333), np.float64(0.037), np.float64(0.004)]
eta = 0.500: [np.float64(3.0), np.float64(2.0), np.float64(2.0), np.float64(2.0)]
eta = 1.000: [np.float64(3.0), np.float64(19.0), np.float64(163.0), np.float64(1459.0)]
eta = 0.52: [np.float64(3.0), np.float64(2.334), np.float64(2.721)]
steps for both coordinates below 0.01 at eta = 1/3: 5


best single step eta = 0.278, f = 0.222


In [5]:
# Why unequal steepness is slow
def run_bowl(curv):
    a, b = 2.0, 2.0 * curv                   # second derivatives of z1^2 and curv*z2^2
    eta = 2 / (a + b)
    z = np.array([1.0, 1.0]); k = 0
    while max(abs(z)) >= 0.01:
        z = z - eta * np.array([2 * z[0], 2 * curv * z[1]]); k += 1
    return (b - a) / (b + a), k

fac, k = run_bowl(50)
print(f"50-to-1 bowl: best factor {fac:.3f}, {k} steps")
assert round(fac, 3) == 0.961 and k == 116
fac, k = run_bowl(2)
print(f"Episode 5 bowl: best factor {fac:.3f}, {k} steps")
assert abs(fac - 1/3) < 1e-12 and k == 5

50-to-1 bowl: best factor 0.961, 116 steps
Episode 5 bowl: best factor 0.333, 5 steps


In [6]:
# Episode 6's toy loss as a bowl: u = (1, 2, 3), y = (2, 4, 5), MSE over (W, beta)
u = np.array([1.0, 2.0, 3.0]); y = np.array([2.0, 4.0, 5.0]); N = 3
H = (2 / N) * np.array([[np.sum(u ** 2), np.sum(u)], [np.sum(u), N]])      # Hessian of the MSE
lam = np.sort(np.linalg.eigvalsh(H))[::-1]
kap = lam / 2
print("second derivatives:", np.round(lam, 3), " kappa:", np.round(kap, 3))
assert np.round(lam, 2).tolist() == [11.09, 0.24] and np.round(kap, 3).tolist() == [5.546, 0.12]
assert np.round(kap[0], 2) == 5.55
factor = (lam[0] - lam[1]) / (lam[0] + lam[1])
steps = int(np.ceil(np.log(0.01) / np.log(factor)))
print(f"best factor {factor:.3f}; about {steps} steps for a 100-fold shrink in distance")
assert round(factor, 3) == 0.958 and steps == 107

second derivatives: [11.093  0.24 ]  kappa: [5.546 0.12 ]
best factor 0.958; about 107 steps for a 100-fold shrink in distance


## Math Lab, Step 4: the 1/N scale
Episode 6 toy, Line A (W = 2, β = 0, N = 3).

In [7]:
def pred(W, b):  return W * u + b
def mse_ab(W, b): return np.mean((y - pred(W, b)) ** 2)
def term_grads(W, b):                     # per-term gradients of (y - W u - b)^2 with respect to (W, b)
    r = y - pred(W, b)
    return np.stack([-2 * r * u, -2 * r], axis=1) + 0.0     # + 0.0 turns -0.0 into 0.0

g = term_grads(2.0, 0.0)
print("per-term gradients (W, beta):", g.tolist())
assert g.tolist() == [[0.0, 0.0], [0.0, 0.0], [6.0, 2.0]]
mean_g, sum_g = g.mean(axis=0), g.sum(axis=0)
assert np.allclose(mean_g, [2, 2 / 3]) and np.allclose(sum_g, [6, 2])

eta = 0.1
new_mean = np.array([2.0, 0.0]) - eta * mean_g
new_sum = np.array([2.0, 0.0]) - eta * sum_g
print("mean step ->", np.round(new_mean, 4), " MSE", round(mse_ab(*new_mean), 4))
assert np.allclose(new_mean, [1.8, -1 / 15]) and abs(mse_ab(*new_mean) - 2 / 15) < 1e-12 and abs(mse_ab(2.0, 0.0) - 1 / 3) < 1e-12
print("sum step  ->", np.round(new_sum, 4), " MSE", round(mse_ab(*new_sum), 4))
assert np.allclose(new_sum, [1.4, -0.2]) and abs(mse_ab(*new_sum) - 1.2) < 1e-12

w = np.array([2.0, 0.0]); seq = [mse_ab(*w)]
for _ in range(3):
    w = w - eta * term_grads(*w).sum(axis=0)
    seq.append(mse_ab(*w))
print("MSE under the summed loss, eta = 0.1:", [round(float(v), 2) for v in seq])
assert [round(seq[1], 2), round(seq[2], 2), round(seq[3], 1)] == [1.2, 5.95, 31.7]      # the episode quotes 31.7 (31.71 to two decimals)

# Episode 6's two-term example: the gradient with respect to the score of "mat" is -(1 - p) = -0.70
assert round(-(1 - 0.30), 2) == -0.70 and round(-(1 - 0.30) / 2, 2) == -0.35

per-term gradients (W, beta): [[0.0, 0.0], [0.0, 0.0], [6.0, 2.0]]
mean step -> [ 1.8    -0.0667]  MSE 0.1333
sum step  -> [ 1.4 -0.2]  MSE 1.2
MSE under the summed loss, eta = 0.1: [0.33, 1.2, 5.95, 31.71]


## Math Lab, Step 5: mini-batch gradients
Draw N′ terms independently and uniformly (with replacement). Exact enumeration, no sampling.

In [8]:
single = term_grads(2.0, 0.0)[:, 0]          # W-component, Line A: 0, 0, 6
print("single-term values:", single.tolist(), " mean", single.mean(), " variance", single.var())
assert single.tolist() == [0, 0, 6] and single.mean() == 2 and single.var() == 8

for n_prime, want_var in ((2, 4), (8, 1)):
    means = [np.mean([single[i] for i in draw]) for draw in product(range(3), repeat=n_prime)]
    print(f"N' = {n_prime}: mean of mini-batch gradients {np.mean(means):.3f}, variance {np.var(means):.3f}")
    assert abs(np.mean(means) - 2) < 1e-9 and abs(np.var(means) - want_var) < 1e-9

# at the least-squares fit (Line B) the full gradient is exactly zero, yet single-term gradients are not
gB = term_grads(1.5, 2 / 3)
print("Line B single-term gradients:", np.round(gB, 4).tolist(), " sum:", np.round(gB.sum(axis=0), 12).tolist())
assert np.allclose(gB, [[1/3, 1/3], [-4/3, -2/3], [1, 1/3]]) and np.allclose(gB.sum(axis=0), 0)

single-term values: [0.0, 0.0, 6.0]  mean 2.0  variance 8.0
N' = 2: mean of mini-batch gradients 2.000, variance 4.000
N' = 8: mean of mini-batch gradients 2.000, variance 1.000
Line B single-term gradients: [[0.3333, 0.3333], [-1.3333, -0.6667], [1.0, 0.3333]]  sum: [0.0, 0.0]


## Check question
f(z) = 3z². (a) Which η converge? (b) From z = 1 with η = 0.1, what are z and f after one step? (c) If the loss were the sum of four copies of f instead of their mean, which η converge, and what happens at η = 0.1?

In [9]:
kappa = 3
print(f"(a) 0 < eta < 1/{kappa} = {1/kappa:.4f}")
z1 = 1 - 2 * 0.1 * kappa * 1
print(f"(b) factor {1 - 2 * 0.1 * kappa:.1f}, z = {z1:.1f}, f = {3 * z1 ** 2:.2f} (down from 3)")
assert round(z1, 1) == 0.4 and round(3 * z1 ** 2, 2) == 0.48
kappa4 = 4 * kappa
print(f"(c) sum of four copies is {kappa4} z^2: eta < 1/{kappa4} = {1/kappa4:.3f}; at eta = 0.1 the factor is {1 - 2 * 0.1 * kappa4:.1f}")
assert round(1 - 2 * 0.1 * kappa4, 1) == -1.4 and abs(1 / kappa4 - 1 / 12) < 1e-12

(a) 0 < eta < 1/3 = 0.3333
(b) factor 0.4, z = 0.4, f = 0.48 (down from 3)
(c) sum of four copies is 12 z^2: eta < 1/12 = 0.083; at eta = 0.1 the factor is -1.4


In [10]:
print("All Episode 7 checks passed.")

All Episode 7 checks passed.
